# Level 4 — Strategic Agent / Workflow Orchestration

This notebook demonstrates the Level 4 Strategic Agent capabilities:

- **UC1**: Goal Decomposition — break a business goal into measurable sub-tasks
- **UC2**: Multi-Agent Orchestration — coordinate Level 2 segmentation + Level 3 campaign
- **UC3**: Revenue Impact Analysis — project revenue from a campaign
- **UC4**: Scenario Simulation — compare strategy options and pick the best
- **UC5**: Pilot Testing — A/B split with compliance gate enforcement
- **UC6**: Roadmap Prioritization — rank initiatives by ROI score
- **UC7**: Self-Reflection Loop — KPI deviation triggers automatic re-plan
- **UC8**: Governance Dashboard — audit trail + Langfuse observability

In [1]:
import sys, os
from pathlib import Path
from utils import *
from src.strategic import get_uc4_scenarios, get_uc6_initiatives
from src.agents.level4 import _decompose

_root = Path(os.getcwd())
if str(_root / 'notebooks') not in sys.path:
    sys.path.insert(0, str(_root)); sys.path.insert(0, str(_root / 'notebooks'))
print('Level 4 Strategic Agent - ready')


Level 4 Strategic Agent - ready


## UC1: Goal Decomposition — Break Business Goal into Sub-Tasks

In [2]:
goal = 'Increase Premium Membership adoption by 5% this quarter'
sub_goals = _decompose(goal)
display_goal_decomposition(sub_goals, goal)


#,Sub-Task
1,1. Develop and implement targeted marketing campaigns to highlight the benefits of Premium Membership.
2,2. Offer limited-time promotions or discounts to incentivise upgrades to Premium Membership.
3,3. Enhance the user experience by adding exclusive features and content for Premium Members.
4,4. Conduct user feedback sessions to identify and address barriers to Premium Membership adoption.


## UC2: Multi-Agent Orchestration — Coordinate Segmentation + Campaign

In [3]:
uc2 = run_uc2_orchestration(goal)
segments, segment_id = uc2['segments'], uc2['segment_id']
estimated_reach, kpi, campaign = uc2['estimated_reach'], uc2['kpi'], uc2['campaign']


Campaign,Date,Reach,Conversions,Opens,Conv. Rate,Open Rate
CAMP-F2E8782F,2025-06-01,65,4,25,6.2%,38.5%
CAMP-3B0A06EB,2025-06-19,52,4,20,7.7%,38.5%
CAMP-6E3FB3CB,2025-07-07,62,4,20,6.5%,32.3%
CAMP-41120329,2025-07-25,45,3,13,6.7%,28.9%
CAMP-3303A8F1,2025-08-12,64,3,18,4.7%,28.1%
CAMP-A1E2EA1F,2025-08-30,65,7,21,10.8%,32.3%
CAMP-A0113BB5,2025-09-17,59,6,24,10.2%,40.7%
CAMP-C1B420E3,2025-10-05,45,5,12,11.1%,26.7%
CAMP-FDE6E9A3,2025-10-23,58,5,16,8.6%,27.6%
CAMP-CB69AE99,2025-11-10,55,3,17,5.5%,30.9%


## UC3: Revenue Impact Analysis — Project Campaign Revenue

In [4]:
from src.tools.strategic import simulate_revenue_impact


impact = simulate_revenue_impact(
    segment_size=estimated_reach,
    conversion_rate=0.08,
    avg_order_value=149.99,
    campaign_cost=500.0,
)
display_revenue_impact(impact)


## UC4: Scenario Simulation — Compare Strategy Options

In [5]:
from src.tools.strategic import run_scenario_simulation


results = run_scenario_simulation(goal, get_uc4_scenarios())
display_scenario_comparison(results)


Scenario,Segment,Conv. Rate,Projected Revenue,Risk,Recommended
Broad Email Blast,all,4.0%,"€1,199.92",medium,
VIP Targeted Offer,vip,18.0%,"€1,599.92",low,✅
At-Risk Win-Back,at_risk,12.0%,€899.91,high,


## UC5: Pilot Testing — A/B Split with Compliance Gate

In [6]:
pilot = run_uc5_pilot(segments)


## UC6: Roadmap Prioritization — Rank Initiatives by ROI

In [7]:
from src.tools.strategic import prioritize_initiatives


ranked = prioritize_initiatives(get_uc6_initiatives())
display_roadmap_priorities(ranked)


Priority,Initiative,Est. Revenue,Effort,ROI Score,Timeline
1,Premium Membership Drive,"€12,000",low,12000.000,Q1
2,Churn Reduction Program,"€15,000",medium,7500.000,Q2
3,Cross-Sell Electronics,"€22,000",high,7333.330,Q2
4,NPS Improvement Campaign,"€5,000",low,5000.000,Q3
5,New Customer Onboarding,"€9,000",medium,4500.000,Q3
6,Win-Back Dormant VIPs,"€8,500",medium,4250.000,Q1


## UC7: Self-Reflection Loop — KPI Deviation Triggers Re-Plan

In [8]:
reflection = run_uc7_reflection(segment_id, goal, kpi, campaign)


## UC8: Governance Dashboard — Audit Trail & Langfuse Observability

### Setting up Langfuse (one-time, ~3 minutes)

Langfuse is a self-hosted observability UI that captures every agent node execution —
latency, token usage, errors, and metadata — as searchable traces.

**Step 1 — Start Langfuse with Docker:**
```bash
# from the project root
cd langfuse
docker compose up -d
# UI is now at http://localhost:3000
```

**Step 2 — Create a project and copy API keys:**
1. Open http://localhost:3000 and sign up (local account, no internet needed)
2. Create a project called `agenticAI`
3. Go to **Settings → API Keys** and copy the Public Key and Secret Key

**Step 3 — Add keys to `.env`:**
```bash
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST=http://localhost:3000
LANGFUSE_PROJECT=customerIntel
```
These are already set in your `.env` — no changes needed.

---

### What you will see in Langfuse

After running the cell below, open http://localhost:3000 → **Traces**.
Each graph invocation creates one trace. Inside it you will find these spans:

| Span name | What it represents | Key metadata |
|---|---|---|
| `orchestrator` | Intent classification + routing decision | `intent`, `confidence` |
| `level4_strategic` | Full strategic orchestration run | `duration_ms`, `tokens_used` |

Each span shows:
- **Duration** — how long the node took in milliseconds
- **Tokens used** — LLM tokens consumed (0 when LLM is unavailable / heuristic fallback)
- **Status** — `DEFAULT` (success) or `ERROR` with the error message
- **Metadata** — `duration_ms` and `tokens_used` as structured fields

If `LANGFUSE_SECRET_KEY` is not set, traces are only stored in the local
in-memory event log (visible in the governance dashboard below) — no error is raised.

In [9]:
run_uc8_langfuse(goal)


{"event": "node_start", "node": "orchestrator", "request_id": "5c0f920b-4ee1-45d0-a867-322ea428dc35", "timestamp": "2026-03-12T13:02:18.726190+00:00"}
{"event": "node_end", "node": "orchestrator", "request_id": "5c0f920b-4ee1-45d0-a867-322ea428dc35", "duration_ms": 590, "tool_calls": 0, "tokens_used": 0, "error": null, "timestamp": "2026-03-12T13:02:19.316927+00:00"}
{"event": "node_start", "node": "level4_strategic", "request_id": "5c0f920b-4ee1-45d0-a867-322ea428dc35", "timestamp": "2026-03-12T13:02:19.623564+00:00"}
{"event": "node_end", "node": "level4_strategic", "request_id": "5c0f920b-4ee1-45d0-a867-322ea428dc35", "duration_ms": 1651, "tool_calls": 1, "tokens_used": 0, "error": null, "timestamp": "2026-03-12T13:02:21.275008+00:00"}


In [10]:
run_uc8_governance(goal, sub_goals, segments, segment_id, campaign, reflection)


timestamp,agent_id,action,user_id
2026-03-12T12:52:12.263932+00:00,orchestrator,classify_intent,manager@shop.com
2026-03-12T12:52:13.196698+00:00,level2_analytics,run_sql_query,manager@shop.com
2026-03-12T12:52:13.959228+00:00,orchestrator,classify_intent,manager@shop.com
2026-03-12T12:52:16.168473+00:00,level1_knowledge,multi_query_search,manager@shop.com
2026-03-12T12:52:18.263493+00:00,orchestrator,classify_intent,manager@shop.com
2026-03-12T12:52:23.878586+00:00,level1_knowledge,multi_query_search,manager@shop.com
2026-03-12T12:52:26.490791+00:00,orchestrator,classify_intent,trustsafety@shop.com
2026-03-12T12:52:26.524140+00:00,level1_knowledge,get_identity_status,trustsafety@shop.com


Node,Duration (ms),Tokens,Error
orchestrator,590,0,—
level4_strategic,1651,0,—
